In [1]:
"""
===============================================================================
AKKADIAN TRANSLATION - KAGGLE TRAINING NOTEBOOK
===============================================================================
Deep Past Challenge: Old Assyrian Cuneiform Translation

This notebook trains a translation model and saves it for inference.
Run this notebook WITH internet enabled to download pre-trained models.

Steps:
1. Upload this as a Kaggle notebook
2. Add the competition data
3. Add your custom dataset (akkadian_combined_weighted.csv, vocab files)
4. Run with GPU enabled
5. Save the output model as a new dataset for inference
===============================================================================
"""

# =============================================================================
# CELL 1: INSTALL DEPENDENCIES
# =============================================================================
# !pip install -q transformers datasets sentencepiece sacrebleu

import os
import pandas as pd
import numpy as np
import re
import json
import warnings
warnings.filterwarnings('ignore')

# Check environment
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================

class Config:
    # Paths - UPDATE THESE FOR YOUR KAGGLE SETUP
    COMPETITION_DATA = "/kaggle/input/deep-past-initiative-machine-translation"  # Competition data
    CUSTOM_DATA = "/kaggle/input/akkadian-training-data"     # Your uploaded dataset
    OUTPUT_DIR = "/kaggle/working/akkadian_model"
    
    # Model settings
    MODEL_NAME = "google/mt5-small"  # or "facebook/nllb-200-distilled-600M"
    
    # Training settings
    MAX_LENGTH = 256
    BATCH_SIZE = 4  # Reduce if OOM
    LEARNING_RATE = 3e-4
    NUM_EPOCHS = 5
    WARMUP_STEPS = 500
    WEIGHT_DECAY = 0.01
    
    # Two-stage training
    STAGE1_EPOCHS = 3  # Pre-train on all data
    STAGE2_EPOCHS = 2  # Fine-tune on OA only
    STAGE2_LR = 1e-4   # Lower LR for fine-tuning

config = Config()

# =============================================================================
# CELL 3: DATA LOADING AND PREPROCESSING
# =============================================================================

def preprocess_akkadian(text):
    """Clean Akkadian transliteration."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    
    # Standardize breaks
    text = re.sub(r'\[\.{3,}\]|\[…\s*…\]|\.{4,}', ' [GAP] ', text)
    text = re.sub(r'x{3,}', ' [GAP] ', text)
    text = re.sub(r'\bx\b', ' [BREAK] ', text)
    text = re.sub(r'\.{3}|…', ' [BREAK] ', text)
    
    # Remove scribal notations but keep content
    text = re.sub(r'[!?]', '', text)
    text = re.sub(r'[/:]', ' ', text)
    text = re.sub(r'[˹˺⌈⌉]', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_english(text):
    """Clean English translation."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def load_competition_data():
    """Load competition training data."""
    train_path = f"{config.COMPETITION_DATA}/train.csv"
    train_df = pd.read_csv(train_path)
    
    print(f"Competition train.csv: {len(train_df)} rows")
    return train_df

def load_sentences_data():
    """Load Old Assyrian sentences (same domain as test!)."""
    sentences_path = f"{config.COMPETITION_DATA}/Sentences_Oare_FirstWord_LinNum.csv"
    
    try:
        sentences_df = pd.read_csv(sentences_path)
        sentences_df = sentences_df[sentences_df['translation'].notna()]
        sentences_df = sentences_df[sentences_df['translation'] != '']
        print(f"OA Sentences: {len(sentences_df)} rows")
        return sentences_df
    except:
        print("Sentences file not found")
        return None

def load_custom_data():
    """Load your uploaded akkademia corpus."""
    custom_path = f"{config.CUSTOM_DATA}/akkadian_combined_weighted.csv"
    
    try:
        custom_df = pd.read_csv(custom_path)
        print(f"Custom weighted data: {len(custom_df)} rows")
        return custom_df
    except:
        print("Custom data not found, will create from scratch")
        return None

def prepare_all_data():
    """Prepare all training data with proper weighting."""
    
    all_data = []
    
    # 1. Competition data (highest priority)
    train_df = load_competition_data()
    for _, row in train_df.iterrows():
        all_data.append({
            'source': preprocess_akkadian(row['transliteration']),
            'target': preprocess_english(row['translation']),
            'domain': 'old_assyrian'
        })
    
    # 2. Sentences data (same domain!)
    sentences_df = load_sentences_data()
    if sentences_df is not None:
        for _, row in sentences_df.iterrows():
            src = row.get('first_word_transcription', '')
            if pd.isna(src):
                src = ''
            all_data.append({
                'source': preprocess_akkadian(src),
                'target': preprocess_english(row['translation']),
                'domain': 'old_assyrian'
            })
    
    # 3. Custom data (if available)
    custom_df = load_custom_data()
    if custom_df is not None:
        # Custom data is already weighted, use directly
        for _, row in custom_df.iterrows():
            all_data.append({
                'source': preprocess_akkadian(row['transliteration']),
                'target': preprocess_english(row['translation']),
                'domain': 'mixed'
            })
    
    df = pd.DataFrame(all_data)
    
    # Filter empty rows
    df = df[(df['source'].str.len() > 0) & (df['target'].str.len() > 0)]
    
    print(f"\nTotal training data: {len(df)} pairs")
    print(f"Old Assyrian: {len(df[df['domain'] == 'old_assyrian'])}")
    print(f"Mixed/Neo-Assyrian: {len(df[df['domain'] == 'mixed'])}")
    
    return df

# Load data
print("="*70)
print("LOADING DATA")
print("="*70)
all_data = prepare_all_data()

# =============================================================================
# CELL 4: CREATE DATASETS FOR TRAINING
# =============================================================================

from transformers import AutoTokenizer
from datasets import Dataset

print("\n" + "="*70)
print("PREPARING TOKENIZER AND DATASETS")
print("="*70)

# Load tokenizer
print(f"\nLoading tokenizer: {config.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

def tokenize_function(examples):
    """Tokenize source and target texts."""
    # Add task prefix for mT5
    if "mt5" in config.MODEL_NAME.lower():
        inputs = ["translate Akkadian to English: " + src for src in examples['source']]
    else:
        inputs = examples['source']
    
    model_inputs = tokenizer(
        inputs,
        max_length=config.MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    labels = tokenizer(
        examples['target'],
        max_length=config.MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Split data
val_size = min(2000, int(len(all_data) * 0.1))
train_data = all_data.iloc[:-val_size].reset_index(drop=True)
val_data = all_data.iloc[-val_size:].reset_index(drop=True)

print(f"Train: {len(train_data)}, Validation: {len(val_data)}")

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_data[['source', 'target']])
val_dataset = Dataset.from_pandas(val_data[['source', 'target']])

# Tokenize
print("Tokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=['source', 'target'])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=['source', 'target'])

print(f"Train dataset: {len(train_dataset)}")
print(f"Val dataset: {len(val_dataset)}")

# =============================================================================
# CELL 5: MODEL TRAINING
# =============================================================================

from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print("\n" + "="*70)
print("TRAINING MODEL")
print("="*70)

# Load model
print(f"\nLoading model: {config.MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(config.MODEL_NAME)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=config.OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=config.LEARNING_RATE,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    num_train_epochs=config.NUM_EPOCHS,
    weight_decay=config.WEIGHT_DECAY,
    warmup_steps=config.WARMUP_STEPS,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",  # Disable wandb
)

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train!
print("\nStarting training...")
trainer.train()

# =============================================================================
# CELL 6: SAVE MODEL FOR INFERENCE
# =============================================================================

print("\n" + "="*70)
print("SAVING MODEL")
print("="*70)

# Save to working directory
save_path = "/kaggle/working/akkadian_final_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

# Also save config for inference
config_dict = {
    'model_name': config.MODEL_NAME,
    'max_length': config.MAX_LENGTH,
}
with open(f"{save_path}/training_config.json", 'w') as f:
    json.dump(config_dict, f)

print("\nFiles saved:")
for f in os.listdir(save_path):
    size = os.path.getsize(f"{save_path}/{f}") / 1e6
    print(f"  {f}: {size:.1f} MB")

# =============================================================================
# CELL 7: TEST INFERENCE
# =============================================================================

print("\n" + "="*70)
print("TESTING INFERENCE")
print("="*70)

def translate(text, model, tokenizer, device=DEVICE):
    """Translate a single text."""
    # Preprocess
    text = preprocess_akkadian(text)
    
    # Add task prefix for mT5
    if "mt5" in config.MODEL_NAME.lower():
        text = "translate Akkadian to English: " + text
    
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", max_length=config.MAX_LENGTH, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=config.MAX_LENGTH,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    
    # Decode
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

# Test examples
test_examples = [
    "KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-{d}IM",
    "a-na 5 GÍN KÙ.BABBAR i-ṣé-er A-šùr-i-mì-tí",
    "um-ma kà-ru-um kà-ni-iš-ma",
]

model.to(DEVICE)

print("\nTest translations:")
for text in test_examples:
    translation = translate(text, model, tokenizer)
    print(f"\n  Input:  {text}")
    print(f"  Output: {translation}")

# =============================================================================
# CELL 8: INSTRUCTIONS FOR NEXT STEPS
# =============================================================================

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("""
1. SAVE THIS MODEL AS A DATASET:
   - Click "Save Version" → "Quick Save"
   - Go to Output tab
   - Click "New Dataset" on the akkadian_final_model folder
   - Name it: "akkadian-translation-model"

2. CREATE INFERENCE NOTEBOOK:
   - Create new notebook for submission
   - Add competition data
   - Add your saved model dataset
   - Turn OFF internet
   - Use the inference code below

3. SUBMIT:
   - Run inference notebook
   - Submit the generated submission.csv
""")

print("\n✅ Training complete!")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4
Using device: cuda
LOADING DATA
Competition train.csv: 1561 rows


OA Sentences: 9772 rows


Custom weighted data: 97371 rows



Total training data: 102478 pairs
Old Assyrian: 10088
Mixed/Neo-Assyrian: 92390



PREPARING TOKENIZER AND DATASETS

Loading tokenizer: google/mt5-small


...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model parameters: 300,176,768



Starting training...


Epoch,Training Loss,Validation Loss
1,0.371400,0.287196
